In [ ]:
import hashlib, random, time, multiprocessing, struct

MESSAGE = b"give my friend 2 bitcoins for a pizza"

def worker(worker_id, found_flag, result_list):
    rng = random.Random()
    while not found_flag.value:
        prefix = struct.pack('>5I',
            rng.randint(0, 0xFFFFFFFF),
            rng.randint(0, 0xFFFFFFFF),
            rng.randint(0, 0xFFFFFFFF),
            rng.randint(0, 0xFFFFFFFF),
            rng.randint(0, 0xFFFFFFFF),
        )
        h = hashlib.sha256(prefix + MESSAGE).hexdigest()
        if h.startswith("00000000"):
            found_flag.value = 1
            result_list.append((prefix.hex(), h))
            return

if __name__ == "__main__":
    num_workers = multiprocessing.cpu_count()
    print(f"Ядер: {num_workers}. Старт!")

    found_flag = multiprocessing.Value('i', 0)
    manager = multiprocessing.Manager()
    result_list = manager.list()

    start = time.time()
    processes = [
        multiprocessing.Process(target=worker, args=(i, found_flag, result_list))
        for i in range(num_workers)
    ]
    for p in processes: p.start()
    for p in processes: p.join()

    elapsed = time.time() - start
    prefix_hex, h = result_list[0]
    print(f"Успіх за {elapsed:.1f}с!")
    print(f"Префікс (HEX): {prefix_hex}")
    print(f"Хеш: {h}")

Ядер: 2. Старт!
Успіх за 5673.4с!
Префікс (HEX): 8e331c382ee12b0487954b678e22a3ce54583a18
Хеш: 00000000134420a740c32c44b49580995bc74745a92588e7c9a3a3c579d80e2a
